### **Student Information**
GitHub ID: janicehuang00

Kaggle name: Jh janice huang

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**
## 1. Model Development (10 pts Required)
**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps
The dataset downloaded from Kaggle includes three files: the tweets, the train/test identification, and the emotion labels. Each of them is indexed by `post_id`, so I simply merged them on `post_id`.
The second step is data cleaning, following the approach in the Master notebook. I cleaned each tweet by converting the text to lowercase, removing URLs and @mentions, and collapsing consecutive spaces, since URLs and user names carry no emotion information. Finally, I split the posts into a training set and a test set according to the given train/test identification.

### 1.2 Feature Engineering Steps
In this step, three types of features are combined: TF-IDF, MiniLM (a pre-trained text model), and pre-trained emotion models.

- **TF-IDF**: I used word 1-2 grams together with character 2-5 grams. Word bigrams capture short phrases (e.g., "not happy"), while character n-grams handle misspelled or stretched words (e.g., "happyyyy"), which are common in short tweets.
- **MiniLM**: As we learned in the Master notebook, word vectors trained on a small dataset are not stable. Therefore, I discarded Word2Vec and used the pre-trained MiniLM models (L6 and L12) introduced in the Master notebook and try to obtain a more stable vector for each post.
- **Pre-trained emotion models**: MiniLM is trained for general text and has no emotion-specific goal, so I also used models trained specifically for emotion classification: j-hartmann/emotion-english-distilroberta-base, SamLowe/roberta-base-go_emotions, and bhadresh-savani/distilbert-base-uncased-emotion. (They are used for inference only.) Just like miniLM, they also turn each post into a vector but based more on emotion, so posts with the same emotion are closer together and different emotions are easier to separate.

In the Master notebook, each representation is evaluated separately. Instead, I concatenated all of the above representations into one large feature matrix (dense vectors are normalized to unit length first) because each one has its own strength. For example, TF-IDF is good at using the exact words, while MiniLM and the emotion models capture the meaning of a post even when its words are unseen in the training data. By combining them, the classifier receives information from multiple perspectives and can predict the correct emotion more successfully.

### 1.3 Explanation of Your Model
I used a Linear SVM as the classifier because the concatenated feature matrix is high-dimensional (many features) while the sample size is small. The reason for choosing it that SVM is simpler (than an MLP), so it is less prone to overfitting. For the hyperparameter, `C=0.03` (a strong regularization) is also meant to prevent overfitting, and `class_weight="balanced"` balances the influence of each class. I chose the hyperparameters with 5-fold cross-validation, measured by macro F1, to avoid overfitting to a single validation split.
Finally, after choosing the hyperparameters, I retrained the model on the whole training set and predicted the test set once to produce the submission.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI
**Collaboration process with the AI assistant**

Some directions came from my own proposals based on the Master notebook, and others from the AI's suggestions. For each one, I asked the AI to explain how it works and judged by myself whether it was reasonable. If it was not clearly unreasonable, the AI wrote the code, I ran the experiments on my own computer, and we discussed the results before choosing the next direction.

- **Baseline (my proposal, from the Master notebook):** I chose the best TF-IDF baseline with a classical classifier. Its public score was below 0.5, so we looked for improvements.
- **Evaluation (AI suggestion):** The AI suggested 5-fold cross-validation instead of a single 20% split. I accepted it because the single split gave a more optimistic score than the public one.
- **TF-IDF tuning (AI suggestion):** The AI suggested word bigrams, character n-grams and `class_weight="balanced"`. I accepted them, and the public score passed the 0.5 baseline.
- **Sentence embeddings (my proposal, from the Master notebook):** I proposed replacing the weak self-trained word vectors with pre-trained MiniLM embeddings next to TF-IDF. Cross-validation improved clearly, while larger encoders gave no clear extra gain.
- **Emotion models (AI suggestion):** The AI suggested using pre-trained emotion classifiers as feature extractors (inference only). I accepted it, and adding them gave the largest improvement.

Every technique that I kept was supported by cross-validation and Kaggle results, not only by the suggestion itself.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
### Add the code related to the preprocessing steps in cells inside this section
# Note: this code was run on my laptop (CPU) with the same steps; it produced submission_v7.csv.
import re, json, os
import numpy as np, pandas as pd

SEED = 42
DATA_DIR = "../competition_data"   # competition files are kept outside the repo

# Load the three competition files and join them on post_id
posts = pd.DataFrame(json.load(open(f"{DATA_DIR}/final_posts.json", encoding="utf-8")))
df = (posts
      .merge(pd.read_csv(f"{DATA_DIR}/data_identification.csv"), on="post_id")          # train / test flag
      .merge(pd.read_csv(f"{DATA_DIR}/emotion.csv"), on="post_id", how="left"))          # labels (train only)

# Light text cleaning: lowercase, remove URLs and @mentions, collapse spaces
def clean_text(s):
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

df["clean"] = df["text"].map(clean_text)

# Split into labelled train posts and unlabelled test posts
is_tr = (df["set"] == "train").values
tr = df[is_tr].reset_index(drop=True)
te = df[~is_tr].reset_index(drop=True)
y = tr["label"].values
tr_pos, te_pos = np.where(is_tr)[0], np.where(~is_tr)[0]   # row positions in df, used to slice feature blocks
print(f"train: {len(tr)}, test: {len(te)}")
print(tr["label"].value_counts())

## 2. Feature Engineering Steps

In [ ]:
### Add the code related to the feature engineering steps in cells inside this section
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# (a) TF-IDF: word 1-2 grams + character 2-5 grams.
# Only defined here; they are fitted inside each training split in Section 3 to avoid data leakage.
def make_tfidf():
    word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    return word_vec, char_vec

# (b) Pretrained sentence vectors (Master Section 5), computed once for all posts and cached
def cached(path, compute):
    if os.path.exists(path) and len(np.load(path)) == len(df):
        return np.load(path)
    M = compute()
    np.save(path, M)
    return M

texts = df["clean"].tolist()
blocks = {}   # name -> dense feature matrix, one row per post in df
blocks["L6"] = cached(f"{DATA_DIR}/minilm_all.npy",
                      lambda: SentenceTransformer("all-MiniLM-L6-v2").encode(texts, show_progress_bar=True))
blocks["L12"] = cached(f"{DATA_DIR}/emb_all-MiniLM-L12-v2.npy",
                       lambda: SentenceTransformer("all-MiniLM-L12-v2").encode(texts, show_progress_bar=True))
print({k: v.shape for k, v in blocks.items()})

In [ ]:
# Emotion-model vectors: pretrained emotion classifiers are used for inference only (no training).
# For each post we keep the first-token vector of the last hidden layer as a sentence feature.
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

EMOTION_MODELS = {
    "emo_v6":  "j-hartmann/emotion-english-distilroberta-base",
    "goemo":   "SamLowe/roberta-base-go_emotions",
    "distil6": "bhadresh-savani/distilbert-base-uncased-emotion",
}
CACHE_FILES = {"emo_v6": "emo_vecs.npy", "goemo": "emo_goemo_vecs.npy", "distil6": "emo_distil6_vecs.npy"}

def emotion_vectors(model_name, texts, batch=32):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i + batch], padding=True, truncation=True, max_length=128, return_tensors="pt")
            hidden = model(**enc, output_hidden_states=True).hidden_states[-1]
            out.append(hidden[:, 0, :].numpy())      # first-token vector of the last layer
    return np.vstack(out)

def unit(M):
    # scale each row to length 1 so every dense block has a comparable size
    return M / np.linalg.norm(M, axis=1, keepdims=True)

for key, name in EMOTION_MODELS.items():
    V = cached(f"{DATA_DIR}/{CACHE_FILES[key]}", lambda: emotion_vectors(name, texts))
    blocks[key] = unit(V)

print({k: v.shape for k, v in blocks.items()})

## 3. Model Implementation Steps

In [ ]:
### Add the code related to the model implementation steps in cells inside this section
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

# Final v7 setting: TF-IDF + MiniLM L6/L12 + three emotion-model vectors, Linear SVM with C=0.03
FEATURE_BLOCKS = ["L6", "L12", "emo_v6", "goemo", "distil6"]
C = 0.03

def build(a_idx, b_idx, txt_a, txt_b, names):
    # Stack TF-IDF (fitted on the training part only) and the dense blocks side by side
    word_vec, char_vec = make_tfidf()
    A = [word_vec.fit_transform(txt_a), char_vec.fit_transform(txt_a)] + [csr_matrix(blocks[n][a_idx]) for n in names]
    B = [word_vec.transform(txt_b), char_vec.transform(txt_b)] + [csr_matrix(blocks[n][b_idx]) for n in names]
    return hstack(A).tocsr(), hstack(B).tocsr()

def make_model():
    return LinearSVC(C=C, class_weight="balanced", random_state=SEED, max_iter=5000)

# 5-fold stratified cross-validation on the labelled posts (macro F1)
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
scores = []
for a, b in skf.split(tr, y):
    Xa, Xb = build(tr_pos[a], tr_pos[b], tr["clean"].iloc[a], tr["clean"].iloc[b], FEATURE_BLOCKS)
    pred = make_model().fit(Xa, y[a]).predict(Xb)
    scores.append(f1_score(y[b], pred, average="macro"))
print(f"5-fold CV macro F1 = {np.mean(scores):.4f} +- {np.std(scores):.4f}")

In [ ]:
# Refit on ALL labelled posts, predict the test posts and write the Kaggle submission
Xa, Xb = build(tr_pos, te_pos, tr["clean"], te["clean"], FEATURE_BLOCKS)
te["label"] = make_model().fit(Xa, y).predict(Xb)

# Kaggle expects two columns: id, label (the sample file calls the id column post_id)
sample = pd.read_csv(f"{DATA_DIR}/sampleSubmission.csv")
sub = (sample[["post_id"]]
       .merge(te[["post_id", "label"]], on="post_id", how="left")
       .rename(columns={"post_id": "id"}))
assert sub["label"].notna().all() and len(sub) == 1440

out = f"{DATA_DIR}/submission_v7.csv"
sub.to_csv(out, index=False)
print("written:", out)
print(sub["label"].value_counts())